# Intro to netcdfs 
<div style="background-color: #f2f9e6; border: 1px solid #d2eeb4; border-radius: 4px; padding: 15px;">

<h3 style="margin-top: 0; background-color: #daeebe; padding: 5px 10px; margin: -15px -15px 15px -15px; border-radius: 4px 4px 0 0; color: #2b542c;">
    ❓ Overview
</h3>  

#### **Questions**
* What is a netcdf? 
* What are the components of working with a netcdf in python?

#### **Objectives**
* Stream a netcdf
* Explore the dimensionality and layers of the netcdf

</div>


## Resources
There are many resources breaking down what a netcdf is, when to use them, and how to work with them in python. Below are just a few, although the internet (and AI like gemini, etc) are extremely helpful should you have any additional burning questions or issues. 
* [Introduction to netcdf](https://adyork.github.io/python-oceanography-lesson/17-Intro-NetCDF/index.html) by data carpentry for oceanographers
  * This resource breaks down WHAT a netcdf is really well, how the data is stored, and the basic components of a netcdf. It has some really great visuals that can be useful for really understanding the layers of arrays in netcdfs.
* [Fundamentals of netcdf storage](https://doc.esri.com/en/arcgis-pro/latest/help/data/multidimensional/fundamentals-of-netcdf-data-storage.html)
  * This resource breaks down the metadata a bit more and gets into the dimensions and attributes of netcdfs.
* [Xarray in 45 minutes](https://tutorial.xarray.dev/overview/xarray-in-45-min.html)
  * This jupyterbook provides a basic coding tutorial of xarray (way more in depth than covered here) 


## Why use netcdfs in oceanography? 
A lot of the data that we will be using is satellite-derived or modeled, which are typically gridded. Netcdfs can efficiently handle large, multidimensional, gridded datasets. 

## How to work with netcdfs in python
[Xarray](https://xarray.dev/), as package in python, makes working with netcdfs incredibly easy, and efficient, and user-friendly. <br>

Lets try opening a netcdf file using Xarray and working with it! We are going to "stream" the data rather than downloading, so you can just run this code as is. ERDDAP wants a certificate verificiation to stream the netcdf. If you try to open the data without explicitly setting ```verify=False```, then you will get an error. This is just a small quirk when pulling netcdf data directly from ERDDAP.

In [1]:
# import packages 
import requests 
import xarray as xr

# load ERDDAP URL 
url = 'https://comet.nefsc.noaa.gov/erddap/griddap/noaa_coastwatch_acspo_v2_nrt.nc?sea_surface_temperature%5B(2026-05-14T12:00:00Z):1:(2026-05-14T12:00:00Z)%5D%5B(34):1:(46)%5D%5B(-77):1:(-63)%5D,sst_gradient_magnitude%5B(2026-05-14T12:00:00Z):1:(2026-05-14T12:00:00Z)%5D%5B(34):1:(46)%5D%5B(-77):1:(-63)%5D'
url_new = requests.get(url,verify=False).content
ds = xr.open_dataset(url_new)

C:\Users\haley.synan\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'comet.nefsc.noaa.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [2]:
ds

<xarray.Dataset> Size: 3MB
Dimensions:                  (time: 1, latitude: 601, longitude: 701)
Coordinates:
  * time                     (time) datetime64[ns] 8B 2026-05-14T12:00:00
  * latitude                 (latitude) float32 2kB 46.01 45.99 ... 34.03 34.01
  * longitude                (longitude) float32 3kB -76.99 -76.97 ... -62.99
Data variables:
    sea_surface_temperature  (time, latitude, longitude) float32 2MB ...
    sst_gradient_magnitude   (time, latitude, longitude) float32 2MB ...
Attributes: (12/63)
    acknowledgement:                        Please acknowledge the use of the...
    aggregator_version:                     V1.00
    cdm_data_type:                          Grid
    col_count:                              18000
    col_start:                              0
    collation_version:                      2.11.0
    ...                                     ...
    summary:                                Sea surface temperature retrieval...
    testOutOfDate:                          now-95days
    time_coverage_end:                      2026-05-14T12:00:00Z
    time_coverage_start:                    2026-05-14T12:00:00Z
    title:                                  Satellite Sea Surface Temperature...
    Westernmost_Easting:                    -76.99

When we are first looking at a netcdf, we can notice a few things. First, there are 3 dimensions: time, latitude, and longitude. Then we can see the netcdf has multiple variables. At this level, we are working with a **xarray dataset**. Note the use of the function ```xr.open_dataset```.


Lets day we want to look at only one of the variables so we can plot it. We can navigate in a layer to look at just one variable in the form of a **xarray data array**. The data array is a wrapper around a numpy array, but with labeled dimensions (in this case, time, latitude, longitude). 

In [3]:
ds.sea_surface_temperature

<xarray.DataArray 'sea_surface_temperature' (time: 1, latitude: 601,
                                             longitude: 701)> Size: 2MB
[421301 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 8B 2026-05-14T12:00:00
  * latitude   (latitude) float32 2kB 46.01 45.99 45.97 ... 34.05 34.03 34.01
  * longitude  (longitude) float32 3kB -76.99 -76.97 -76.95 ... -63.01 -62.99
Attributes:
    colorBarMaximum:        35.0
    colorBarMinimum:        0.0
    comment:                SST obtained by regression with buoy measurements...
    coverage_content_type:  physicalMeasurement
    ioos_category:          Temperature
    long_name:              sea surface sub-skin temperature
    source:                 NOAA
    standard_name:          sea_surface_subskin_temperature
    units:                  degree_C
    valid_max:              40.0
    valid_min:              -2.0

This example netcdf only shows 1 timestep, but we can navigate into this layer using ```[0]```. This is asking for the first time step (or only in this case). You can see that the time dimension is no longer availabe, only latitude and longitude. *NOTE: Python starts counting at 0, not 1 like Matlab or some other languages* 

In [4]:
ds.sea_surface_temperature[0]

<xarray.DataArray 'sea_surface_temperature' (latitude: 601, longitude: 701)> Size: 2MB
[421301 values with dtype=float32]
Coordinates:
    time       datetime64[ns] 8B 2026-05-14T12:00:00
  * latitude   (latitude) float32 2kB 46.01 45.99 45.97 ... 34.05 34.03 34.01
  * longitude  (longitude) float32 3kB -76.99 -76.97 -76.95 ... -63.01 -62.99
Attributes:
    colorBarMaximum:        35.0
    colorBarMinimum:        0.0
    comment:                SST obtained by regression with buoy measurements...
    coverage_content_type:  physicalMeasurement
    ioos_category:          Temperature
    long_name:              sea surface sub-skin temperature
    source:                 NOAA
    standard_name:          sea_surface_subskin_temperature
    units:                  degree_C
    valid_max:              40.0
    valid_min:              -2.0

As mentioned, a data array is a wrapper for a **numpy array.** By adding a ```.values```, we can now access the dataset (latitude and longitude) as a numpy array. Here we can see that this is a 2-D array.  If we wanted to work with the data in this format, we would need to use numpy functions NOT xarray functions.

In [5]:
ds.sea_surface_temperature[0].values

array([[  nan,   nan,   nan, ...,  5.86,  6.16,  5.95],
       [  nan,   nan,   nan, ...,   nan,   nan,   nan],
       [  nan,   nan,   nan, ...,   nan,   nan,   nan],
       ...,
       [23.42, 23.42, 23.43, ...,   nan,   nan,   nan],
       [23.41, 23.45, 23.46, ...,   nan,   nan,   nan],
       [23.43, 23.43, 23.45, ...,   nan,   nan,   nan]], dtype=float32)

Lets say we now wanted to flatten these values to 1-D, rather than 2-D. 

In [6]:
ds.sea_surface_temperature[0].flatten()

AttributeError: 'DataArray' object has no attribute 'flatten'

Hmmmm... that doesn't work... lets try that again.

In [14]:
ds.sea_surface_temperature[0].values.flatten()

array([nan, nan, nan, ..., nan, nan, nan], dtype=float32)

Why would ```.flatten()``` work in the second code chunk, but not the first? [.flatten()](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.flatten.html) is a numpy function, not an xarray function. Remember, we can convert a xarray dataarray to a numpy array using ```.data``` or ```.values```. While an easy fix, its important to keep in mind what data type you are working with and the dimensionality when choosing the function to apply. Python has fairly good error handling, but understanding the meaning of the errors can take some getting used to. You can check the data type using the built in ```type()``` function. 


In [17]:
type(ds.sea_surface_temperature[0].values)

numpy.ndarray

Thats all for now! Let me know if you have any questions regarding netcdfs, xarray functions, or documentation. 